In [46]:
import pandas as pd
import numpy as np

import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))

from src.utils import limpiar_df

pd.set_option('display.max_columns', None)

In [47]:
ruta = '../data/raw/fbref_big5'

In [48]:
RAW = Path(ruta)

In [49]:
for carpeta in sorted(RAW.iterdir()):
    if not carpeta.is_dir():
        continue
    temporada = carpeta.name          # '2021-2022'
    for archivo in sorted(carpeta.glob("*.parquet")):
        print(temporada, archivo.stem)

2021-2022 big5_2021-2022_misc
2021-2022 big5_2021-2022_playingtime
2021-2022 big5_2021-2022_shooting
2021-2022 big5_2021-2022_stats
2022-2023 big5_2022-2023_misc
2022-2023 big5_2022-2023_playingtime
2022-2023 big5_2022-2023_shooting
2022-2023 big5_2022-2023_stats
2023-2024 big5_2023-2024_misc
2023-2024 big5_2023-2024_playingtime
2023-2024 big5_2023-2024_shooting
2023-2024 big5_2023-2024_stats
2024-2025 big5_2024-2025_misc
2024-2025 big5_2024-2025_playingtime
2024-2025 big5_2024-2025_shooting
2024-2025 big5_2024-2025_stats
2025-2026 big5_2025-2026_misc
2025-2026 big5_2025-2026_playingtime
2025-2026 big5_2025-2026_shooting
2025-2026 big5_2025-2026_stats


In [50]:
por_temporada = []
CLAVE = ['Player', 'Squad', 'Comp', 'Born']

for carpeta in sorted(RAW.iterdir()):
    if not carpeta.is_dir():
        continue
    temporada = carpeta.name

    # 1. cargar las cuatro tablas de esta temporada
    tablas = {}
    for archivo in carpeta.glob("*.parquet"):
        tipo = archivo.stem.split("_")[-1]      # 'stats', 'misc', ...
        tablas[tipo] = limpiar_df(pd.read_parquet(archivo))
    # 2. unirlas entre sí  -> df_temporada
    # ...  aquí va tu lógica de merge
    

    base = tablas['stats'].copy()

    for tipo in ['shooting', 'misc', 'playingtime']:
        otra = tablas[tipo]
        nuevas = [c for c in otra.columns if c not in base.columns]
        base = base.merge(
            otra[CLAVE + nuevas],
            on=CLAVE,
            how='left',
            validate='one_to_one'
        )

    df_temporada = base

    df_temporada["Season"] = temporada
    por_temporada.append(df_temporada)

# 3. apilar todas las temporadas
completo = pd.concat(por_temporada, ignore_index=True)

In [51]:
completo.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_PKwon,Performance_PKcon,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,21,2000,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,32.0,0,13,2,15.4,0.41,0.06,0.00,0.00,0,0,0,26,52,1,50,26,45,NaN,NaN,1,85,84.2,32,89,29,2,22,2,0.53,18,69,-51,-1.59,+0.08,2021-2022
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,33,1987,34,34,2983,33.1,2,0,2,2,0,0,5,1,0.06,0.00,0.06,0.06,0.06,33.1,2,18,6,33.3,0.54,0.18,0.11,0.33,0,0,0,38,25,0,4,67,28,NaN,NaN,0,88,87.2,34,88,33,0,NaN,0,1.12,38,41,-3,-0.09,-0.50,2021-2022
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,21,2000,31,29,2462,27.4,1,0,1,1,0,0,12,3,0.04,0.00,0.04,0.04,0.04,27.4,1,18,5,27.8,0.66,0.18,0.06,0.20,0,0,2,43,37,0,14,40,26,NaN,NaN,0,79,72.0,29,82,19,2,38,1,1.10,26,43,-17,-0.62,+0.69,2021-2022
3,4,Laurent Abergel,FRA,MF,Lorient,Ligue 1,28,1993,34,34,2956,32.8,0,2,2,0,0,0,9,0,0.00,0.06,0.06,0.00,0.06,32.8,0,30,7,23.3,0.91,0.21,0.00,0.00,0,0,0,41,69,1,37,54,63,NaN,NaN,0,87,86.4,34,87,26,0,NaN,0,0.82,27,59,-32,-0.97,-1.75,2021-2022
4,5,Charles Abi,FRA,FW,Saint-Étienne,Ligue 1,21,2000,1,1,45,0.5,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.5,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,0,1,0,0,1,0,0,NaN,NaN,0,45,1.3,1,45,0,0,NaN,0,1.00,0,0,0,0.00,+0.93,2021-2022


Correcion del tipo de dato

In [52]:
completo.drop(columns = ['Performance_PKwon', 'Performance_PKcon'], inplace = True)
completo['Team Success_+/-'] = completo['Team Success_+/-'].str.replace('+', '', regex = False)
completo['Team Success_+/-90'] = completo['Team Success_+/-90'].str.replace('+', '', regex = False)
completo['Team Success_On-Off'] = completo['Team Success_On-Off'].str.replace('+', '', regex = False)
vars_numericas = ['Age', 'Born', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A',
                  'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G+A-PK',
                  '90s', 'Standard_Gls', 'Standard_Sh', 'Standard_SoT', 'Standard_SoT%', 'Standard_Sh/90', 'Standard_SoT/90', 'Standard_G/Sh', 'Standard_G/SoT', 'Standard_PK', 'Standard_PKatt',
                  'Performance_2CrdY', 'Performance_Fls', 'Performance_Fld', 'Performance_Off', 'Performance_Crs', 'Performance_Int', 'Performance_TklW', 'Performance_OG',
                  'Playing Time_Mn/MP', 'Playing Time_Min%', 'Starts_Starts', 'Starts_Mn/Start', 'Starts_Compl', 'Subs_Subs', 'Subs_Mn/Sub', 'Subs_unSub',
                  'Team Success_PPM', 'Team Success_onG', 'Team Success_onGA', 'Team Success_+/-', 'Team Success_+/-90', 'Team Success_On-Off']
completo[vars_numericas] = completo[vars_numericas].apply(pd.to_numeric, errors='coerce')

In [53]:
completo.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,Playing Time_MP,Playing Time_Starts,Playing Time_Min,Playing Time_90s,Performance_Gls,Performance_Ast,Performance_G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,90s,Standard_Gls,Standard_Sh,Standard_SoT,Standard_SoT%,Standard_Sh/90,Standard_SoT/90,Standard_G/Sh,Standard_G/SoT,Standard_PK,Standard_PKatt,Performance_2CrdY,Performance_Fls,Performance_Fld,Performance_Off,Performance_Crs,Performance_Int,Performance_TklW,Performance_OG,Playing Time_Mn/MP,Playing Time_Min%,Starts_Starts,Starts_Mn/Start,Starts_Compl,Subs_Subs,Subs_Mn/Sub,Subs_unSub,Team Success_PPM,Team Success_onG,Team Success_onGA,Team Success_+/-,Team Success_+/-90,Team Success_On-Off,Season
0,1,Max Aarons,ENG,DF,Norwich City,Premier League,21.0,2000.0,34,32,2881,32.0,0,2,2,0,0,0,8,0,0.00,0.06,0.06,0.00,0.06,32.0,0,13,2,15.4,0.41,0.06,0.00,0.00,0,0,0,26,52,1,50,26,45,1,85,84.2,32,89.0,29,2,22.0,2,0.53,18,69,-51,-1.59,0.08,2021-2022
1,2,Yunis Abdelhamid,MAR,DF,Reims,Ligue 1,33.0,1987.0,34,34,2983,33.1,2,0,2,2,0,0,5,1,0.06,0.00,0.06,0.06,0.06,33.1,2,18,6,33.3,0.54,0.18,0.11,0.33,0,0,0,38,25,0,4,67,28,0,88,87.2,34,88.0,33,0,NaN,0,1.12,38,41,-3,-0.09,-0.50,2021-2022
2,3,Salis Abdul Samed,GHA,MF,Clermont Foot,Ligue 1,21.0,2000.0,31,29,2462,27.4,1,0,1,1,0,0,12,3,0.04,0.00,0.04,0.04,0.04,27.4,1,18,5,27.8,0.66,0.18,0.06,0.20,0,0,2,43,37,0,14,40,26,0,79,72.0,29,82.0,19,2,38.0,1,1.10,26,43,-17,-0.62,0.69,2021-2022
3,4,Laurent Abergel,FRA,MF,Lorient,Ligue 1,28.0,1993.0,34,34,2956,32.8,0,2,2,0,0,0,9,0,0.00,0.06,0.06,0.00,0.06,32.8,0,30,7,23.3,0.91,0.21,0.00,0.00,0,0,0,41,69,1,37,54,63,0,87,86.4,34,87.0,26,0,NaN,0,0.82,27,59,-32,-0.97,-1.75,2021-2022
4,5,Charles Abi,FRA,FW,Saint-Étienne,Ligue 1,21.0,2000.0,1,1,45,0.5,0,0,0,0,0,0,0,0,0.00,0.00,0.00,0.00,0.00,0.5,0,0,0,NaN,0.00,0.00,NaN,NaN,0,0,0,1,0,0,1,0,0,0,45,1.3,1,45.0,0,0,NaN,0,1.00,0,0,0,0.00,0.93,2021-2022
